# 05 - Apply eligibility, tailored segments (All / Overseas / Indian capped / Indian uncapped), phase-wise stats
Inputs: outputs/*_ratings.csv (notebook 04), outputs/balls_pressure.parquet (notebook 02), data/player_eligibility_1.csv
Output: outputs/pressure_players_phase_stats.xlsx

In [5]:
import numpy as np, pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter as L

pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 100)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'

# newest player_eligibility*.csv in data/ is used automatically (rename-proof)
cands = sorted(DATA.glob('player_eligibility*.csv'), key=lambda p: p.stat().st_mtime)
if not cands:
    raise FileNotFoundError('Put your eligibility CSV in %s (file name must start with player_eligibility)' % DATA)
ELIG = cands[-1]
print('eligibility file used:', ELIG.name)
for f in ['batter_ratings.csv', 'bowler_ratings.csv', 'balls_pressure.parquet']:
    if not (OUT / f).exists():
        raise FileNotFoundError('%s is missing in %s - run notebooks 01 to 04 first' % (f, OUT))

E = pd.read_csv(ELIG, encoding='utf-8-sig')               # utf-8-sig survives Excel's "CSV UTF-8" BOM
E.columns = [c.strip().lower() for c in E.columns]
need = ['player_id', 'name', 'team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team']
if [c for c in need if c not in E.columns]:
    raise KeyError('Eligibility CSV is missing columns: %s (found: %s)' % ([c for c in need if c not in E.columns], list(E.columns)))
E['player_id'] = pd.to_numeric(E.player_id, errors='coerce')
E = E.dropna(subset=['player_id'])
E['player_id'] = E.player_id.astype(int)
for c in ['team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team']:
    E[c] = E[c].astype(str).str.strip().str.lower().replace({'nan': np.nan, '': np.nan})
blank = E.current_ipl_team.isna().sum()
if blank:
    print('WARNING: %d blank current_ipl_team values treated as "available"' % blank)
E['current_ipl_team'] = E.current_ipl_team.fillna('available')
dups = E.player_id.duplicated(keep='last').sum()
if dups:
    print('WARNING: %d duplicate player_id rows - keeping the last one' % dups)
E = E.drop_duplicates('player_id', keep='last').set_index('player_id')

# Safeguard 1: anyone who has batted or bowled for Pakistan / Pakistan A is excluded, whatever the file says.
P0 = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=['striker_id', 'bowler_id', 'bat_team', 'bowl_team'])
PAK = set(P0[P0.bat_team.str.contains('Pakistan')].striker_id) | set(P0[P0.bowl_team.str.contains('Pakistan')].bowler_id)
print('Pakistan-linked ids found in ball data:', len(PAK), '| of which present in eligibility file:', sorted(PAK & set(E.index)))

# Safeguard 2: id 6922 is the PAKISTAN Abdul Samad; the Indian LSG player is id 7446 (row moved to the right id).
if 7446 not in E.index and 6922 in E.index:   # only needed for the older file version
    E.loc[7446] = E.loc[6922]
    E.loc[7446, 'name'] = 'Abdul Samad'

E['status'] = np.select([E.index.isin(PAK), E.current_ipl_team.eq('retired'), E.current_ipl_team.eq('available')],
                        ['excluded (Pakistan)', 'retired', 'no IPL team'], 'has IPL team')
ECOLS = ['team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team', 'status']


def load(path):
    T = pd.read_csv(path, index_col=0)
    T.index.name = 'player_id'
    T = T.join(E[ECOLS], how='left')
    T['status'] = T.status.fillna('excluded (not in eligibility file)')
    return T


BAT, BOWL = load(OUT / 'batter_ratings.csv'), load(OUT / 'bowler_ratings.csv')
if (BAT.consensus.notna() & ~BAT.eligible & (BAT.raw_p < 50)).sum() == 0:
    print('WARNING: ratings CSV looks like an older notebook 04 output - rerun notebook 04 first')
print('batters by status :', BAT.status.value_counts().to_dict())
print('bowlers by status :', BOWL.status.value_counts().to_dict())

eligibility file used: player_eligibility_1.csv
Pakistan-linked ids found in ball data: 65 | of which present in eligibility file: []
batters by status : {'no IPL team': 428, 'has IPL team': 193, 'excluded (not in eligibility file)': 77, 'retired': 37}
bowlers by status : {'no IPL team': 320, 'has IPL team': 151, 'excluded (not in eligibility file)': 54, 'retired': 19}


## Segments: everyone below has NO confirmed IPL team, is not retired and has never played for Pakistan
All | Overseas | Indian capped | Indian uncapped - ranked inside each segment on the same consensus score as notebook 04.
Evidence grade: A = meets sample threshold and CI above zero | B = meets threshold, CI includes zero | C = thin sample (30-59 pressure balls or < 8 pressure innings)

In [6]:
ACTIVE = ['has IPL team', 'no IPL team']
POOL = lambda T: T.status.eq('no IPL team')
SEGMENTS = {
    'All': lambda T: POOL(T),
    'Overseas': lambda T: POOL(T) & T.indian_or_overseas.eq('overseas'),
    'Indian capped': lambda T: POOL(T) & T.indian_or_overseas.eq('indian') & T.capped_indian.eq('yes'),
    'Indian uncapped': lambda T: POOL(T) & T.indian_or_overseas.eq('indian') & T.capped_indian.eq('no'),
}


def prep(T):
    T = T.copy()
    T['grade'] = np.select([T.eligible & (T.ci_lo > 0), T.eligible], ['A', 'B'], 'C')
    act = T[T.status.isin(ACTIVE) & T.consensus.notna()].sort_values('consensus', ascending=False)
    T['rank_active'] = pd.Series(np.arange(1, len(act) + 1), index=act.index)
    return T


def segment(T, seg):
    d = T[SEGMENTS[seg](T) & T.consensus.notna()].sort_values('consensus', ascending=False).copy()
    d['rank_segment'] = np.arange(1, len(d) + 1)
    return d


BAT, BOWL = prep(BAT), prep(BOWL)
SEG = {(seg, role): segment(T, seg) for seg in SEGMENTS for role, T in (('batters', BAT), ('bowlers', BOWL))}
bat_act = BAT[BAT.rank_active.notna()].sort_values('rank_active')
bowl_act = BOWL[BOWL.rank_active.notna()].sort_values('rank_active')

SHOW = ['name', 'team', 'grade', 'raw_p', 'inn_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'pl_dot', 'consensus', 'pressure_type']
for (seg, role), d in SEG.items():
    print('\n=== %s - %s: %d scored | grades %s' % (seg.upper(), role.upper(), len(d), d.grade.value_counts().sort_index().to_dict()))
    print(d.head(8)[['rank_segment', 'rank_active'] + SHOW].round(1).to_string())


=== ALL - BATTERS: 130 scored | grades {'A': 1, 'B': 37, 'C': 92}
           rank_segment  rank_active             name         team grade  raw_p  inn_p  alpha    pl  ci_lo  ci_hi  pl_dot  consensus pressure_type
player_id                                                                                                                                         
2379                  1         19.0      Ben Duckett      england     C     61    6.0   11.4  14.1   -1.1   29.3     4.1        1.0       Neutral
6280                  2         21.0      Harry Brook      england     C     81    7.0   10.8  13.6    2.8   24.4     2.9        1.0       Neutral
22565                 3         26.0  Muhammad Waseem   associates     B    115   10.0    7.5  10.2   -6.9   27.4     0.8        0.8       Neutral
2436                  4         30.0    Dasun Shanaka    sri lanka     A    257   16.0   10.9  13.7    0.9   26.4    -0.3        0.7       Neutral
4106                  5         34.0   Daryl Mitche

## Phase-wise counts from ball-by-ball (powerplay 1-6, middle 7-15, death 16-20; regulation innings only)
Batting: runs off the bat, balls faced (wides excluded, no-balls included), dismissals incl. run-outs, retired-out excluded.
Bowling: legal balls, runs conceded (bat runs + wides + no-balls), wickets credited to the bowler (no run-outs).

In [7]:
P = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=[
    'innings', 'phase', 'striker_id', 'bowler_id', 'runs', 'faced', 'legal', 'bowler_runs', 'bowler_wkt',
    'dismissed_player_id', 'dismissal_type', 'is_wicket', 'match_tier'])
P = P[P.innings.isin([1, 2])]
PH = ['PP', 'MID', 'DEATH']

bt = P[P.faced].groupby(['striker_id', 'phase']).agg(balls=('runs', 'size'), runs=('runs', 'sum'))
outs = P[P.is_wicket & P.dismissal_type.ne('Retire Out')].groupby(['dismissed_player_id', 'phase']).size().rename('outs')
outs.index.names = ['striker_id', 'phase']
BATP = bt.join(outs, how='outer').fillna(0).unstack('phase').reindex(columns=PH, level=1).fillna(0)

bw = P.groupby(['bowler_id', 'phase']).agg(balls=('legal', 'sum'), runs=('bowler_runs', 'sum'), wkts=('bowler_wkt', 'sum'))
BOWLP = bw.unstack('phase').reindex(columns=PH, level=1).fillna(0)

weak = P.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])
weak_bat = P[P.faced & weak].groupby('striker_id').size() / P[P.faced].groupby('striker_id').size()
weak_bowl = P[weak].groupby('bowler_id').size() / P.groupby('bowler_id').size()

## Excel export (counts are values; SR / average / economy are live formulas)

In [8]:
F = Font(name='Arial', size=10)
HF = Font(name='Arial', size=10, bold=True, color='FFFFFF')
HFILL = PatternFill('solid', start_color='1F3864')
BFILL = PatternFill('solid', start_color='D9E1F2')
wb = Workbook()


def write_table(ws, df, cols, headers, start_row=1, number_formats=None):
    for j, h in enumerate(headers, 1):
        c = ws.cell(row=start_row, column=j, value=h)
        c.font, c.fill, c.alignment = HF, HFILL, Alignment(horizontal='center', vertical='center', wrap_text=True)
    for i, (_, r) in enumerate(df[cols].iterrows(), start_row + 1):
        for j, col in enumerate(cols, 1):
            v = r[col]
            v = None if (isinstance(v, float) and np.isnan(v)) else (v.item() if hasattr(v, 'item') else v)
            c = ws.cell(row=i, column=j, value=v)
            c.font = F
            if number_formats and col in number_formats and v is not None:
                c.number_format = number_formats[col]
    ws.freeze_panes = ws.cell(row=start_row + 1, column=3)
    for j in range(1, len(cols) + 1):
        ws.column_dimensions[L(j)].width = 14 if j > 2 else (8 if j == 1 else 24)
    ws.row_dimensions[start_row].height = 32


NF = {'rank_active': '0', 'raw_p': '0', 'inn_p': '0', 'alpha': '0.0', 'pl': '0.0', 'ci_lo': '0.0', 'ci_hi': '0.0', 'consensus': '0.00', 'weak_share': '0%',
      'pl_dot': '0.0', 'pl_bnd': '0.0', 'pl_ext': '0.0', 'sr_idx': '0', 'dot_p': '0.0', 'dot_exp_p': '0.0', 'bd_ratio_p': '0.00', 'rdi_death': '0.00', 'rdi_n': '0',
      'dot_rise': '0.0', 'dot_rise_exp': '0.0', 'chain_break': '0.0', 'chain_break_exp': '0.0', 'chain_n': '0', 'dot_pct_p': '0.0', 'over_dot_pct_p': '0.0',
      'p_overs': '0', 'econ_p': '0.00', 'ext_per_over_p': '0.00', 'ext_exp_over_p': '0.00', 'ext_runs_p': '0'}
RC = ['name', 'team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team', 'raw_p', 'inn_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'consensus', 'pressure_type', 'weak_share']
RH = ['Player', 'National team', 'Indian / overseas', 'Capped (Indian)', 'IPL team', 'Pressure balls', 'Pressure innings', 'Baseline skill', 'Pressure rating', 'CI low', 'CI high',
      'Consensus score', 'Pressure type (indicative)', 'Share of pressure balls vs weak tiers']
XC = {'bat': (['pl_dot', 'pl_bnd', 'sr_idx', 'dot_p', 'dot_exp_p', 'bd_ratio_p', 'rdi_death', 'rdi_n', 'dot_rise', 'dot_rise_exp', 'chain_break', 'chain_break_exp', 'chain_n'],
              ['Dot control (pp)', 'Boundary rate (pp)', 'SR index (pressure, 100 = expected)', 'Dot % (pressure)', 'Expected dot %', 'Boundary:dot ratio',
               'Chase-death rate delivery (1 = as required)', 'Chase-death balls', 'Dot % while RRR rising', 'Expected dot %', 'Chain-break % (after 2+ dots, RRR rising)', 'Expected %', 'Chain balls']),
      'bowl': (['pl_dot', 'pl_ext', 'dot_pct_p', 'dot_exp_p', 'over_dot_pct_p', 'p_overs', 'econ_p', 'ext_per_over_p', 'ext_exp_over_p', 'ext_runs_p'],
               ['Dot-ball rate (pp)', 'Extras avoided (pp)', 'Dot % (pressure)', 'Expected dot %', 'Over-level dot % (pressure overs)', 'Pressure overs', 'Economy (pressure)',
                'Extras per over', 'Expected extras per over', 'Extra runs conceded (pressure)'])}


def rank_sheet(name, d, rank_col='rank_segment', role='bat'):
    ws = wb.create_sheet(name)
    d = d.reset_index()
    xc, xh = XC[role]
    tail = [] if rank_col == 'rank_active' else ['rank_active']
    write_table(ws, d, [rank_col, 'name', 'grade'] + RC[1:] + xc + tail,
                ['Rank' if rank_col == 'rank_active' else 'Rank in segment', 'Player', 'Evidence grade'] + RH[1:] + xh + ['Rank (all active)'] * len(tail),
                number_formats=NF)
    ws.auto_filter.ref = ws.dimensions


ws0 = wb.active
ws0.title = 'README'
notes = [
    'Pressure players - tailored segments (All / Overseas / Indian capped / Indian uncapped), batters and bowlers',
    'Source: notebooks 01-05 on ball_by_ball.csv, matches.csv and the player_eligibility*.csv supplied by the user (newest file in data/ is used).',
    'Pool = player is in the eligibility file, current_ipl_team is "available" (no confirmed team), not retired, and has never played for Pakistan / Pakistan A in the ball data.',
    'Segments: Overseas = indian_or_overseas "overseas". Indian capped / uncapped = indian_or_overseas "indian" with capped_indian "yes" / "no".',
    'To refresh after trades, releases or retentions: edit current_ipl_team in the CSV, rerun notebook 05. Nothing else changes.',
    'Pressure = top 20% of leverage within competition tier (innings 2: win-probability swing; innings 1: projected score vs par). Super overs count as maximum pressure.',
    'Pressure rating, baseline skill and CI are runs-equivalent per 100 balls vs an average player in the same match state (empirical-Bayes shrunk).',
    'Rank = consensus score across 12 specifications (pressure threshold 70/80/90, weak-tier weight 0.5/1.0, consistency weight 0.1/0). Everyone with >=30 pressure balls is scored.',
    'Evidence grade: A = >=60 pressure balls, >=8 pressure innings and CI above zero. B = meets the sample threshold, CI includes zero. C = thin sample (30-59 balls or <8 innings): treat as a lead, not a ranking.',
    'Style metrics (percentage points per 100 balls vs an average player in the same state): batters = dot control (fewer dots) and boundary rate; bowlers = dot-ball rate and extras avoided. Each is shrunk like the main rating and counts in the score only as far as it is repeatable.',
    'Pressure profile: SR index = strike rate on pressure balls / expected x 100. Boundary:dot ratio is smoothed (+1). Chase-death rate delivery = runs scored / runs required over his chase balls in overs 16-20 (contest alive; 1.0 = exactly as required). Dot % while RRR rising and chain-break % use chase balls where the required rate rose over the last 6 balls.',
    'Extras are charged in full: a wide costs its runs, a no-ball costs its runs plus the free-hit premium (about 0.9 runs), so conceding extras under pressure lowers a bowler\'s rating.',
    'Phase stats: regulation innings of IPL and T20I data from 2022 on, incl. associate and A-team games (see "vs weak tiers" share). PP = overs 1-6, MID = 7-15, DEATH = 16-20.',
    'Batting: SR = runs / balls faced x 100; Avg = runs / dismissals (run-outs included, retired-out excluded); "-" = not out in that phase.',
    'Bowling: Econ = runs per over; Avg = runs per wicket; SR = balls per wicket (run-outs not credited to the bowler); "-" = no wickets.',
    'Pressure type (Thrives / Dips) is indicative only: the pressure-vs-baseline gap had low split-half reliability.',
]
for i, t in enumerate(notes, 1):
    c = ws0.cell(row=i, column=1, value=t)
    c.font = Font(name='Arial', size=12 if i == 1 else 10, bold=(i == 1))
ws0.column_dimensions['A'].width = 170

rows = []
for (seg, role), d in SEG.items():
    g = d.grade.value_counts()
    rows.append({'segment': seg, 'role': role, 'scored': len(d), 'A': int(g.get('A', 0)), 'B': int(g.get('B', 0)), 'C': int(g.get('C', 0)),
                 'top3': ', '.join(d.head(3).name)})
SUMM = pd.DataFrame(rows)
wsu = wb.create_sheet('Segment summary')
write_table(wsu, SUMM, list(SUMM.columns), ['Segment', 'Role', 'Players scored', 'Grade A', 'Grade B', 'Grade C', 'Top 3'])
wsu.column_dimensions['G'].width = 60

for seg in SEGMENTS:
    rank_sheet('%s - batters' % seg, SEG[(seg, 'batters')], role='bat')
    rank_sheet('%s - bowlers' % seg, SEG[(seg, 'bowlers')], role='bowl')
rank_sheet('Ref - top batters (all active)', bat_act.head(25), 'rank_active', 'bat')
rank_sheet('Ref - top bowlers (all active)', bowl_act.head(25), 'rank_active', 'bowl')


def phase_sheet(title, ids, roles):
    ws = wb.create_sheet(title)
    bat = roles == 'bat'
    base = ['Player', 'National team', 'IPL team']
    metrics = ['Balls', 'Runs', 'Outs', 'SR', 'Avg'] if bat else ['Balls', 'Runs', 'Wkts', 'Econ', 'Avg', 'SR']
    blocks = ['Overall'] + ['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)']
    ncols = len(metrics)
    for j, h in enumerate(base, 1):
        ws.cell(row=2, column=j, value=h)
    for k, blk in enumerate(blocks):
        c0 = 4 + k * ncols
        ws.cell(row=1, column=c0, value=blk)
        ws.merge_cells(start_row=1, start_column=c0, end_row=1, end_column=c0 + ncols - 1)
        for m, h in enumerate(metrics):
            ws.cell(row=2, column=c0 + m, value=h)
    last = 4 + len(blocks) * ncols
    ws.cell(row=2, column=last, value='% balls vs weak tiers')
    for row in ws.iter_rows(min_row=1, max_row=2, max_col=last):
        for cell in row:
            cell.font, cell.fill, cell.alignment = HF, HFILL, Alignment(horizontal='center', vertical='center', wrap_text=True)
    T = BAT if bat else BOWL
    DATAP = BATP if bat else BOWLP
    for i, pid in enumerate(ids, 3):
        ws.cell(row=i, column=1, value=T.loc[pid, 'name'])
        ws.cell(row=i, column=2, value=T.loc[pid, 'team'])
        ws.cell(row=i, column=3, value=T.loc[pid, 'current_ipl_team'])
        cnt = DATAP.loc[pid] if pid in DATAP.index else None
        for k in range(len(blocks)):
            c0 = 4 + k * ncols
            if bat:
                vals = [(cnt['balls', ph] if cnt is not None else 0) for ph in PH], [(cnt['runs', ph] if cnt is not None else 0) for ph in PH], [(cnt['outs', ph] if cnt is not None else 0) for ph in PH]
            else:
                vals = [(cnt['balls', ph] if cnt is not None else 0) for ph in PH], [(cnt['runs', ph] if cnt is not None else 0) for ph in PH], [(cnt['wkts', ph] if cnt is not None else 0) for ph in PH]
            if k == 0:   # overall = sum of the three phase blocks (live formulas)
                for m in range(3):
                    refs = '+'.join('%s%d' % (L(4 + (p + 1) * ncols + m), i) for p in range(3))
                    ws.cell(row=i, column=c0 + m, value='=' + refs)
            else:
                for m in range(3):
                    ws.cell(row=i, column=c0 + m, value=float(vals[m][k - 1]))
            B, R_, W = (L(c0 + x) + str(i) for x in range(3))
            if bat:
                ws.cell(row=i, column=c0 + 3, value='=IF(%s=0,"",%s/%s*100)' % (B, R_, B))
                ws.cell(row=i, column=c0 + 4, value='=IF(%s=0,"-",%s/%s)' % (W, R_, W))
            else:
                ws.cell(row=i, column=c0 + 3, value='=IF(%s=0,"",%s/%s*6)' % (B, R_, B))
                ws.cell(row=i, column=c0 + 4, value='=IF(%s=0,"-",%s/%s)' % (W, R_, W))
                ws.cell(row=i, column=c0 + 5, value='=IF(%s=0,"-",%s/%s)' % (W, B, W))
        wk = (weak_bat if bat else weak_bowl).get(pid, 0)
        ws.cell(row=i, column=last, value=float(wk) if wk == wk else 0.0)
        for j in range(1, last + 1):
            cell = ws.cell(row=i, column=j)
            cell.font = F
            if j > 3 and j < last:
                cell.number_format = '0.0' if ((j - 4) % ncols) >= 3 else '0'
            if j == last:
                cell.number_format = '0%'
    ws.freeze_panes = 'D3'
    ws.column_dimensions['A'].width = 24
    ws.column_dimensions['B'].width = 14
    ws.column_dimensions['C'].width = 26
    for j in range(4, last + 1):
        ws.column_dimensions[L(j)].width = 8
    ws.column_dimensions[L(last)].width = 12
    ws.row_dimensions[2].height = 32
    return ws


bat_ids = list(dict.fromkeys(i for seg in SEGMENTS for i in SEG[(seg, 'batters')].head(15).index))
bowl_ids = list(dict.fromkeys(i for seg in SEGMENTS for i in SEG[(seg, 'bowlers')].head(15).index))
phase_sheet('Phase stats - batters', bat_ids, 'bat')
phase_sheet('Phase stats - bowlers', bowl_ids, 'bowl')
target = OUT / 'pressure_players_phase_stats.xlsx'
try:
    wb.save(target)
except PermissionError:                                   # the file is open in Excel
    target = OUT / ('pressure_players_phase_stats_%s.xlsx' % pd.Timestamp.now().strftime('%H%M%S'))
    wb.save(target)
    print('Excel had the old file open - saved under a new name instead')
print('\nsaved', target, '| batters in phase sheet:', len(bat_ids), '| bowlers:', len(bowl_ids))


saved d:\Projects\superr\outputs\pressure_players_phase_stats.xlsx | batters in phase sheet: 28 | bowlers: 30
